# nb_validacion_final
| Campo | Valor |
|---|---|
| Archivo | `04_validacion/nb_validacion_final.ipynb` |
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Validación · **última tarea del job** `job_andina_diario` |
| Propósito | Comprobar que la corrida **hizo su trabajo**, no solo que terminó. Si una prueba no se cumple, el notebook lanza un error: el job queda en rojo y llega la alerta por correo |
| Entradas / Salidas | Lee `bronze`, `silver` y `gold` del catálogo del ambiente. No escribe nada |
| Decisiones | Cada prueba tiene una condición de aprobación explícita (falsable). Corre con `max_retries: 0`: reintentar no arregla un dato malo. El análisis detallado y exploratorio sigue en `nb_validacion_silver_gold` |
| Idempotencia | Solo lectura: re-ejecutarlo no cambia nada |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 1.0 · 2026-11-13 · primera versión como tarea del job |

In [0]:
dbutils.widgets.text("catalogo", "andina_dev", "Catálogo (ambiente)")
CAT = dbutils.widgets.get("catalogo").strip()

resultados = []   # (prueba, aprobada, detalle)

def registrar(prueba: str, aprobada: bool, detalle: str) -> None:
    resultados.append((prueba, bool(aprobada), detalle))
    print(f"{'✓' if aprobada else '✗'} {prueba}: {detalle}")

def uno(sql: str):
    """Primera fila de una consulta (con {cat} reemplazado por el catálogo del ambiente)."""
    return spark.sql(sql.format(cat=CAT)).first()

print(f"Validando el catálogo {CAT}")

## 1. Hay datos en cada capa
Una corrida que termina "bien" con tablas vacías es la falla silenciosa más común (por ejemplo, credenciales que leen una base vacía).

In [0]:
TABLAS_NO_VACIAS = ["bronze.orders_raw", "silver.clientes", "silver.pedidos", "silver.pagos",
                    "silver.items_pedido", "gold.fact_ventas", "gold.fact_pagos", "gold.dim_clientes"]
for t in TABLAS_NO_VACIAS:
    n = uno(f"SELECT count(*) AS n FROM {{cat}}.{t}").n
    registrar(f"no vacía · {t}", n > 0, f"{n:,} filas")

## 2. Frescura de bronze
La última ingesta debe ser de los últimos 2 días. Si no, el job "corrió" pero la extracción no trajo nada nuevo en mucho tiempo.

In [0]:
r = uno("SELECT max(_ingestion_ts) AS ultima, max(_ingestion_ts) >= current_timestamp() - INTERVAL 2 DAYS AS fresca FROM {cat}.bronze.orders_raw")
registrar("frescura · bronze.orders_raw", r.fresca, f"última ingesta {r.ultima}")

## 3. Conciliación bronze → silver + cuarentena
Por entidad: claves vivas en bronze (último cambio distinto de DELETE) = filas en silver + claves en cuarentena. Es la prueba de **«nada se descarta en silencio»**.

In [0]:
conciliacion = spark.sql('''
WITH ultimo AS (
  SELECT 'clientes' tabla, CustomerID k, _op, row_number() OVER (PARTITION BY CustomerID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) rn FROM {cat}.bronze.customers_raw
  UNION ALL SELECT 'productos', ProductID, _op, row_number() OVER (PARTITION BY ProductID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.products_raw
  UNION ALL SELECT 'pedidos', OrderID, _op, row_number() OVER (PARTITION BY OrderID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.orders_raw
  UNION ALL SELECT 'items_pedido', OrderItemID, _op, row_number() OVER (PARTITION BY OrderItemID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.order_items_raw
  UNION ALL SELECT 'pagos', PaymentID, _op, row_number() OVER (PARTITION BY PaymentID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.payments_raw
  UNION ALL SELECT 'tickets', TicketID, _op, row_number() OVER (PARTITION BY TicketID ORDER BY coalesce(_commit_ts, UpdatedAt) DESC, _lsn DESC, _seqval DESC) FROM {cat}.bronze.support_tickets_raw),
esperado AS (SELECT tabla, count(*) claves_vivas FROM ultimo WHERE rn = 1 AND _op <> 'D' GROUP BY tabla),
real AS (
  SELECT 'clientes' tabla, count(*) filas FROM {cat}.silver.clientes UNION ALL SELECT 'productos', count(*) FROM {cat}.silver.productos
  UNION ALL SELECT 'pedidos', count(*) FROM {cat}.silver.pedidos UNION ALL SELECT 'items_pedido', count(*) FROM {cat}.silver.items_pedido
  UNION ALL SELECT 'pagos', count(*) FROM {cat}.silver.pagos UNION ALL SELECT 'tickets', count(*) FROM {cat}.silver.tickets),
cuarentena AS (SELECT tabla, count(DISTINCT clave) en_cuarentena FROM {cat}.silver.cuarentena GROUP BY tabla)
SELECT e.tabla, e.claves_vivas, coalesce(c.en_cuarentena, 0) AS en_cuarentena, r.filas AS silver_filas
FROM esperado e JOIN real r USING (tabla) LEFT JOIN cuarentena c USING (tabla) ORDER BY 1'''.format(cat=CAT)).collect()

for f in conciliacion:
    registrar(f"conciliación · {f.tabla}", f.claves_vivas - f.en_cuarentena == f.silver_filas,
              f"bronze {f.claves_vivas:,} − cuarentena {f.en_cuarentena:,} = silver {f.silver_filas:,}")

## 4. Clave única en silver
AUTO CDC garantiza una fila por clave; esta prueba lo confirma en cada corrida (un cambio de configuración podría romperlo sin aviso).

In [0]:
CLAVES = {"clientes": "id_cliente", "productos": "id_producto", "pedidos": "id_pedido",
          "items_pedido": "id_item", "pagos": "id_pago", "tickets": "id_ticket"}
for tabla, clave in CLAVES.items():
    r = uno(f"SELECT count(*) - count(DISTINCT {clave}) AS repetidas FROM {{cat}}.silver.{tabla}")
    registrar(f"clave única · silver.{tabla}", r.repetidas == 0, f"{r.repetidas} claves repetidas")

## 5. SCD2: una sola versión vigente por clave
En las tablas con historia, cada clave tiene exactamente una versión abierta (`__END_AT` nulo). Dos vigentes significa que un análisis "al día de hoy" contaría doble.

In [0]:
HISTORIALES = {"clientes_historial": "id_cliente", "pagos_historial_estados": "id_pago"}
for tabla, clave in HISTORIALES.items():
    r = uno(f"""SELECT count(*) AS claves_con_varias FROM (
                   SELECT {clave} FROM {{cat}}.silver.{tabla} WHERE __END_AT IS NULL
                   GROUP BY {clave} HAVING count(*) > 1)""")
    registrar(f"una vigente · silver.{tabla}", r.claves_con_varias == 0, f"{r.claves_con_varias} claves con más de una vigente")

## 6. Gold cuadra con silver
`fact_ventas` debe tener las mismas líneas y el mismo monto que las líneas de silver con pedido; `fact_pagos`, los mismos pagos.

In [0]:
r = uno('''SELECT (SELECT count(*) FROM {cat}.gold.fact_ventas) AS lineas_gold,
                  (SELECT count(*) FROM {cat}.silver.items_pedido l JOIN {cat}.silver.pedidos p USING (id_pedido)) AS lineas_silver,
                  (SELECT round(sum(monto_item), 2) FROM {cat}.gold.fact_ventas) AS ventas_gold,
                  (SELECT round(sum(l.monto_item), 2) FROM {cat}.silver.items_pedido l JOIN {cat}.silver.pedidos p USING (id_pedido)) AS ventas_silver,
                  (SELECT count(*) FROM {cat}.gold.fact_pagos) AS pagos_gold,
                  (SELECT count(*) FROM {cat}.silver.pagos) AS pagos_silver''')
registrar("gold = silver · líneas de venta", r.lineas_gold == r.lineas_silver, f"gold {r.lineas_gold:,} · silver {r.lineas_silver:,}")
registrar("gold = silver · monto de ventas", r.ventas_gold == r.ventas_silver, f"gold {r.ventas_gold} · silver {r.ventas_silver}")
registrar("gold = silver · pagos", r.pagos_gold == r.pagos_silver, f"gold {r.pagos_gold:,} · silver {r.pagos_silver:,}")

## 7. Veredicto
Si alguna prueba falló, se lanza un error con la lista: la tarea queda fallida, el job en rojo y llega el correo de alerta.

In [0]:
fallidas = [(p, d) for p, ok, d in resultados if not ok]
print(f"\n{len(resultados) - len(fallidas)} de {len(resultados)} pruebas aprobadas en {CAT}")
if fallidas:
    raise AssertionError(f"{len(fallidas)} prueba(s) de validación fallaron en {CAT}:\n"
                         + "\n".join(f"  ✗ {p}: {d}" for p, d in fallidas))